# Denoising Autoencoder on FIDS30

Train a convolutional autoencoder to reconstruct clean fruit images from noisy inputs.
Images are resized to 128×128 for computational efficiency.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

IMG_SIZE = 256
BATCH_SIZE = 320
EPOCHS = 30
LR = 1e-3
NOISE_STD = 0.15

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_ds = datasets.ImageFolder("PrepData/FIDS30/Training", transform=transform)
val_ds = datasets.ImageFolder("PrepData/FIDS30/Validation", transform=transform)
test_ds = datasets.ImageFolder("PrepData/FIDS30/Test", transform=transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

## 1. Convolutional Autoencoder

In [ ]:
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder: 128 -> 64 -> 32 -> 16
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),   # -> 64
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  # -> 32
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), # -> 16
            nn.ReLU(),
        )
        # Decoder: 16 -> 32 -> 64 -> 128
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=1), # -> 32
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),  # -> 64
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 3, stride=2, padding=1, output_padding=1),   # -> 128
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

model = ConvAutoencoder().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")
print(model)

## 2. Training

Add Gaussian noise to inputs, train to reconstruct the clean image (MSE loss).

In [ ]:
def add_noise(images, std=NOISE_STD):
    noisy = images + std * torch.randn_like(images)
    return noisy.clamp(0, 1)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = {"train_loss": [], "val_loss": []}

for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss = 0.0
    for images, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False):
        images = images.to(device)
        noisy = add_noise(images)
        recon = model(noisy)
        loss = criterion(recon, images)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        train_loss += loss.item()

    # Validate
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, _ in val_loader:
            images = images.to(device)
            noisy = add_noise(images)
            recon = model(noisy)
            val_loss += criterion(recon, images).item()

    history["train_loss"].append(train_loss / len(train_loader))
    history["val_loss"].append(val_loss / len(val_loader))
    print(f"Epoch {epoch+1}: Train Loss={history['train_loss'][-1]:.6f}, Val Loss={history['val_loss'][-1]:.6f}")

torch.save(model.state_dict(), "denoising_ae.pth")
print("Model saved to denoising_ae.pth")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(history["train_loss"])+1), history["train_loss"], label="Train", marker='o', ms=3)
ax.plot(range(1, len(history["val_loss"])+1), history["val_loss"], label="Val", marker='o', ms=3)
ax.set(xlabel="Epoch", ylabel="MSE Loss", title="Denoising Autoencoder: Training Curves")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Denoising Results

Show noisy input → reconstruction → clean original for test samples.

In [ ]:
model.eval()
images, _ = next(iter(test_loader))
images = images.to(device)
noisy = add_noise(images)
with torch.no_grad():
    recon = model(noisy)

n_show = min(8, len(images))
fig, axes = plt.subplots(3, n_show, figsize=(2.5 * n_show, 7.5))
for i in range(n_show):
    axes[0, i].imshow(noisy[i].cpu().permute(1, 2, 0).clamp(0, 1))
    axes[0, i].axis('off')
    axes[1, i].imshow(recon[i].cpu().permute(1, 2, 0).clamp(0, 1))
    axes[1, i].axis('off')
    axes[2, i].imshow(images[i].cpu().permute(1, 2, 0).clamp(0, 1))
    axes[2, i].axis('off')

axes[0, 0].set_ylabel("Noisy Input", fontsize=11)
axes[1, 0].set_ylabel("Reconstruction", fontsize=11)
axes[2, 0].set_ylabel("Original", fontsize=11)
plt.suptitle(f"Denoising Autoencoder (noise σ={NOISE_STD})", fontsize=13)
plt.tight_layout()
plt.show()